In [ ]:
import os
import fitz  # PyMuPDF
import pandas as pd
import numpy as np
import easyocr
from PIL import Image
import re
import nltk
from nltk.corpus import stopwords

# 1. Inisialisasi OCR (Biar aman, kita matiin dulu GPU-nya kalau takut crash)
reader = easyocr.Reader(['id', 'en'], gpu=False) 

def extract_teks_cv(path_pdf):
    full_text = ""
    try:
        doc = fitz.open(path_pdf)
        
        # Coba baca teks digital dulu
        for page in doc:
            full_text += page.get_text()
            
        # JURUS ANTI-88 KARAKTER:
        # Kalau teksnya dikit banget, berarti ini PDF GAMBAR. Langsung pake OCR.
        if len(full_text.strip()) < 200:
            print("🚨 PDF terdeteksi sebagai gambar. Menjalankan OCR...")
            full_text = ""
            for page in doc:
                pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
                img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
                result = reader.readtext(np.array(img), detail=0)
                full_text += " ".join(result) + "\n"
        
        doc.close()
        return full_text
    except Exception as e:
        return f"Error: {e}"

# --- TEST SEKARANG ---
path_cv = "CVYT.pdf" # Pastiin filenya ada di folder yang sama
hasil = extract_teks_cv(path_cv)

print(f"DEBUG: Karakter yang dapet: {len(hasil)}")
print("-" * 20)
print(hasil[:500])

In [ ]:
%pip install easyocr

In [ ]:
import os
print("📍 Isha lagi ada di folder:", os.getcwd())
print("📂 File yang Isha liat di folder ini:", os.listdir())

In [ ]:
%pip install openai

In [ ]:
import accelerate
print(accelerate.__version__)

In [ ]:
def extract_text_from_pdf(file_path):
    try:
        with open(file_path, 'rb') as file:
            reader = PyPDF2.PdfReader(file)
            text = ""
            for page in reader.pages:
                text += page.extract_text()
            return text
    except Exception as e:
        return f"Error: {e}"

def clean_text_pro(text):
    # [cite_start]Lowercase & hapus karakter selain huruf [cite: 3, 5]
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    
    # [cite_start]Buang Stopwords (kata umum) biar fokus ke skill [cite: 98]
    stop_words = set(stopwords.words('english'))
    words = text.split()
    meaningful_words = [w for w in words if w not in stop_words]
    
    return " ".join(meaningful_words)

In [ ]:
%pip install chromadb sentence-transformers
import chromadb
from chromadb.utils import embedding_functions
import os

# 1. Inisialisasi Chroma (Lokal di folder 'db_smartpath')
chroma_client = chromadb.PersistentClient(path="./db_smartpath")

# 2. Pakai embedding model yang ringan tapi pinter (jalan di CPU/GPU)
default_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

# 3. Bikin 'Collection' (ibaratnya tabel di database)
# Kalau sudah ada, dia bakal ngambil yang lama
collection = chroma_client.get_or_create_collection(name="resumes", embedding_function=default_ef)

def ingest_all_categories(base_path):
    existing_ids = collection.get()['ids']
    for category_name in os.listdir(base_path):
        category_path = os.path.join(base_path, category_name)
        if os.path.isdir(category_path):
            print(f"\n📂 Memproses kategori: {category_name}")
            # looping dimulai
            for file_name in os.listdir(category_path):
                if file_name.endswith(".pdf"):
                    # ceki ceki biar gaada doppelgangger(duplikat)
                    if file_name in existing_ids:
                        continue

                    try:
                        path_lengkap = os.path.join(category_path, file_name)
                        # extract and clean text
                        raw_text = extract_text_from_pdf(path_lengkap)
                        cleaned_text = clean_text_pro(raw_text)
                        # masukin ke Chroma
                        collection.add(
                            documents=[cleaned_text],
                            metadatas=[{"filename": file_name, "category": category_name}],
                            ids=[file_name]
                        )
                        print(f"✅ {file_name} WE DID IT!")
                    except Exception as e:
                        print(f"❌ Gagal proses {file_name}: {e}")

    print(f"\n----- MANTAP GASII BESTI udah ada {collection.count()} dokumen di database guwwa -----")

ingest_all_categories(r"D:\dicoding bootcamp\project capstone\data\resume dataset\data")

In [ ]:
def cari_kandidat(lowongan_teks, jumlah_hasil=5):
    lowongan_bersih = clean_text_pro(lowongan_teks)
    # tanya pada database cuyy
    hasil = collection.query(
        query_texts=[lowongan_bersih],
        n_results=jumlah_hasil
    )

    print(f"Hasil Pencarian untuk: '{lowongan_teks[:50]}...' \n")
    for i in range(len(hasil['ids'][0])):
        print(f"Ranking {i+1}: {hasil['ids'][0][i]}")
        print(f"Kemiripan: {round((1 - hasil['distances'][0][i]) * 100, 2)}%\n")
    return hasil

# contoh cara pake
# masukin kriteria
lowongan_saya = "di butuhkan data analyst yang jago SQL, Python, dan bisa bikin dashboard pake Tableau"
cari_kandidat(lowongan_saya)

In [ ]:
pip install pdfplumber

In [ ]:
# Jalankan ini di cell baru
%pip install -U accelerate bitsandbytes transformers

In [ ]:
from dotenv import load_dotenv
import os
from huggingface_hub import login

load_dotenv("hugging_face_token.env")
access_token = os.getenv("HF_TOKEN")

if access_token:
    login(token=access_token)
else:
    print("Waduh Bes, tokennya gak ketemu! Cek file .env ya.")

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16,
    llm_int8_enable_fp32_cpu_offload=True
)
# setting agar pake GPU
model_id = "Qwen/Qwen2.5-3B-Instruct"

print("--loading euy kalem, sabar, tenang, lagi masukin ke otak si isha ke GPU--")
tokenizer = AutoTokenizer.from_pretrained(model_id, token=access_token)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto", # auto pake CUDA maksudnya kl ada
    trust_remote_code=True,
    token = access_token
)

def analisis_cv_user(path_pdf_user, teks_loker):
    print(f"---memproses CV kamu agar peluang kamu lebih besar 📂: {path_pdf_user}---")
    # ekstrak dan cleaning
    try:
        raw_text = extract_text_from_pdf(path_pdf_user)
        teks_cv_user =  clean_text_pro(raw_text)
    except Exception as e:
        print(f"❌ gagal memproses bes, coba lagi lain kali yaa: {e}")
        return
    # rakit prompt "two sided" buat mesin gua
    massages = [
        {"role": "system", "content": "Kamu adalah Konsultan Karir AI. Jawab dalam Bahasa Indonesia yang profesional namun suportif. Gunakan format yang diminta secara kaku."},
        {"role": "user", "content": f"""Analisis kecocokan CV saya dengan loker: {teks_loker}
         
    TEKS CV SAYA:
    {teks_cv_user[:2000]}

    BALAS DENGAN PEMBAGIAN FORMAT SEBAGAI BERIKUT:
, 
    [SISI KIRI: ANALISIS KECOCOKAN CV]
    - Skor keseluruhan kecocokan (0-100%):
    - 3 hard skills yang sudah cocok:
    - 3 soft skills yang sudah cocok:
    - Alasan utama kenapa saya layak diposisi ini:

    [SISI KANAN: UPGRADE SKILL UNTUK MENAMBAH PELUANG]
    - Skill yang masih kurang untuk posisi dan alasan kenapa skill itu penting [GAP ANALYSIS]:
    - Rekomendasi 3 kursus online spesifik untuk belajar skill tersebut (misal: "Belajar SQL di Coursera: [link kursus]"):
    - Tips jitu mandraguna buat benerin CV saya agar lebih 'ATS-friendly' dan menarik perhatian HR:

    Berikan skor akhir di baris paling bawah dengan format 'TOTAL SKOR: XX%' berdasarkan analisis kamu.
    """}
    ]

    # ayo mikir mesin, jadi mesinnn
    text_prompt = tokenizer.apply_chat_template(massages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text_prompt, return_tensors="pt").to("cuda")

    print("--Sabar Bes, Isha lagi mikir keras...--")
    outputs = model.generate(
        **inputs,
        max_new_tokens=1000,
        temperature=0.2,
        top_p=0.9,
        repetition_penalty=1.1
    )

    input_length = inputs['input_ids'].shape[1]
    jawaban_isha = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
    # tampilan output (simulasi kiri kanan, o kotak)
    print("\n" + "="*60)
    print("Hasil REVIEW KARIR OLEH ASISTEN ISHA KESAYANGAN ANDA :V")
    print("="*60)
    # bagi output berdasarkan mark yang sudah ditentukan
    print(jawaban_isha)
    print("="*60)

# --- CARA PAKAI (EKSEKUSI) ---
# Misal kamu simpen CV kamu di folder project dengan nama 'CV_Saya.pdf'
loker_tujuan = "Data Scientist yang paham Machine Learning, Python, dan SQL"
analisis_cv_user("Aditya Wardhana CV.pdf", loker_tujuan)

In [ ]:
import pdfplumber

def extract_text_from_pdf(pdf_path):
    full_text = ""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                text = page.extract_text()
                if text:
                    full_text += text + "\n"
        return full_text
    except Exception as e:
        return f"Waduh error pas baca PDF: {e}"

# KITA LIHAT HASILNYA:
hasil_tes = extract_text_from_pdf("CVYT.pdf")
print(f"DEBUG: Teks yang kebaca sekarang ada {len(hasil_tes)} karakter.")

In [ ]:
%pip install pymupdf

In [ ]:
import fitz
import os

def extract_text_from_pdf(nama_file):
    # Ini buat dapetin alamat lengkap folder kamu sekarang
    folder_sekarang = os.getcwd()
    path_lengkap = os.path.join(folder_sekarang, nama_file)
    
    print(f"DEBUG: Nyari file di: {path_lengkap}")
    
    if not os.path.exists(path_lengkap):
        return f"ERROR: File beneran gak ada di {path_lengkap} Bes!"

    try:
        doc = fitz.open(path_lengkap)
        text = ""
        for page in doc:
            text += page.get_text("text") + "\n"
        doc.close() # Tutup filenya biar gak memory leak
        return text
    except Exception as e:
        return f"ERROR: Gagal baca karena {e}"

# --- COBA TEST LAGI ---
teks_cv = extract_text_from_pdf("CVYT.pdf")
print(f"JUMLAH KARAKTER: {len(teks_cv)}")
if len(teks_cv) > 200:
    print("MANTAP! Teks udah kebaca ribuan karakter!")
    print(teks_cv[:300])
else:
    print("Waduh, kok masih dikit? Coba cek filenya di Canva, 'Save as' lagi tapi pilih 'PDF Standard' Bes!")

In [ ]:
import fitz  # Ini library PyMuPDF

def extract_text_from_pdf(pdf_path):
    text = ""
    try:
        # Buka dokumen
        doc = fitz.open(pdf_path)
        for page in doc:
            # Ambil teks secara kasar (Raw Text)
            # 'blocks' biasanya lebih ampuh buat layout Canva
            text += page.get_text("text") + "\n"
        return text
    except Exception as e:
        return f"Waduh error: {e}"

# --- TEST LAGI SEKARANG ---
hasil_tes = extract_text_from_pdf("CVYT.pdf") # Pake CV kamu dulu
print(f"DEBUG: Teks yang kebaca sekarang ada {len(hasil_tes)} karakter.")
print("--- Isinya: ---")
print(hasil_tes[:300])

In [ ]:
from openai import OpenAI
import os

# Konek ke Local Server LM Studio
client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

def analisis_cv_user(path_pdf_user, teks_loker):
    print(f"--- Memproses CV 📂: {path_pdf_user} ---")
    
    # Proses ekstraksi (pastikan fungsi extract_cv_super_power dan clean_text_pro sudah di-run sebelumnya)
    try:
        raw_text = extract_cv_super_power(path_pdf_user)
        teks_cv_user = clean_text_pro(raw_text)
    except Exception as e:
        print(f"❌ Gagal memproses PDF: {e}")
        return

    messages = [
        {"role": "system", "content": "Kamu adalah Konsultan Karir AI. Jawab dalam Bahasa Indonesia yang profesional namun suportif. Gunakan format yang diminta secara kaku."},
        {"role": "user", "content": f"""Analisis kecocokan CV saya dengan loker: {teks_loker}
         
    TEKS CV SAYA:
    {teks_cv_user[:2000]}

    BALAS DENGAN PEMBAGIAN FORMAT SEBAGAI BERIKUT:
    
    [SISI KIRI: ANALISIS KECOCOKAN CV]
    - Skor keseluruhan kecocokan (0-100%):
    - 3 hard skills yang sudah cocok:
    - 3 soft skills yang sudah cocok:
    - Alasan utama kenapa saya layak diposisi ini:

    [SISI KANAN: UPGRADE SKILL UNTUK MENAMBAH PELUANG]
    - Skill yang masih kurang untuk posisi dan alasan kenapa skill itu penting [GAP ANALYSIS]:
    - Rekomendasi 3 kursus online spesifik untuk belajar skill tersebut:
    - Tips jitu mandraguna buat benerin CV saya agar lebih 'ATS-friendly' dan menarik perhatian HR:

    Berikan skor akhir di baris paling bawah dengan format 'TOTAL SKOR: XX%' berdasarkan analisis kamu.
    """}
    ]

    print("-- Mengirim data ke LM Studio... Sabar ya! --")
    
    try:
        response = client.chat.completions.create(
            model="qwen2.5-3b-instruct",
            messages=messages,
            temperature=0.2,
            max_tokens=1000,
        )
        
        jawaban_isha = response.choices[0].message.content
        
        print("\n" + "="*60)
        print("Hasil REVIEW KARIR OLEH ASISTEN ISHA")
        print("="*60)
        print(jawaban_isha)
        print("="*60)
        
    except Exception as e:
        print(f"❌ Gagal konek ke LM Studio. Pastikan server lokal sudah nyala! Error detail: {e}")

# --- CARA PAKAI (EKSEKUSI) ---
# Misal kamu simpen CV kamu di folder project dengan nama 'CV_Saya.pdf'
loker_tujuan = "pekerja part time di restoran cepat saji yang ramah dan cekatan, bisa bekerja dalam tim, dan siap bekerja dengan shift malam"
analisis_cv_user("CVYT.pdf", loker_tujuan)

In [ ]:
import sys
print(sys.executable)

In [ ]:
import fitz # PyMuPDF
import easyocr
import numpy as np
from PIL import Image

# Inisialisasi Reader (Cukup sekali aja biar gak berat)
reader = easyocr.Reader(['id', 'en']) 

def extract_cv_super_power(pdf_path):
    # 1. Cek apakah file ada
    if not os.path.exists(pdf_path):
        return f"❌ File {pdf_path} tidak ditemukan. Pastikan file ada di folder yang benar."
    
    # 2. Coba baca teks digital dulu (Cara Cepet)
    doc = fitz.open(pdf_path)
    full_text = ""
    for page in doc:
        full_text += page.get_text()
    
    # 3. Kalau hasilnya "gaji honorer" (< 200 karakter), pake OCR!
    if len(full_text.strip()) < 200:
        print("🔍 Isha mendeteksi ini PDF Gambar. Menyalakan Mata OCR...")
        full_text = ""
        for page_num in range(len(doc)):
            page = doc.load_page(page_num)
            pix = page.get_pixmap()
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            # Baca gambar jadi teks
            result = reader.readtext(np.array(img), detail=0)
            full_text += " ".join(result) + "\n"
    
    doc.close()
    return full_text

# --- TEST LAGI SEKARANG BES ---
teks_cv = extract_cv_super_power("Aditya Wardhana CV.pdf")
print(f"DEBUG FINAL: Karakter kebaca {len(teks_cv)}. GAK MUNGKIN 88 LAGI!")